# Modell 2 – Multinomial Naive Bayes och modelljämförelse

I denna notebook tränar vi en Multinomial Naive Bayes-modell för att klassificera Amazon-recensioner som positiva, negativa eller neutrala.

Vi jämför modellen med logistisk regression från steg 3. Båda modellerna använder samma träningsdata, testdata och TF-IDF-features för att jämförelsen ska bli rättvis.

In [1]:
import pandas as pd
import re
import joblib as joblib

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
# Läser in samma data som i 03.ipynb filen
reviews = pd.read_csv(
    "../data/processed/all_beauty_sample_100k.csv",
    keep_default_na=False
)

print("Antal recensioner:", len(reviews))

Antal recensioner: 100000


In [3]:

# Rensa text på samma sätt som tidigare
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

reviews["clean_text"] = reviews["text"].apply(preprocess_text)

reviews[["text", "clean_text"]].head()

,text,clean_text
0,"love this lip balm, it’s very minting and crea...",love this lip balm it s very minting and creat...
1,I was so happy to have these on hand after my ...,i was so happy to have these on hand after my ...
2,"Great product, worth the money",great product worth the money
3,Great products and great customer service,great products and great customer service
4,Didnt like that It eventually breaks off after...,didnt like that it eventually breaks off after...


### Dela upp träningsdata och testdata

In [4]:

# Samma uppdelning som i 03.ipynb filen
train_reviews, test_reviews = train_test_split(
    reviews,
    test_size=0.2,
    random_state=42,
    stratify=reviews["sentiment"]
)

print("Träningsdata:", len(train_reviews))
print("Testdata:", len(test_reviews))


Träningsdata: 80000
Testdata: 20000


### Ladda TF-IDF och skapa features

In [5]:
# Ladda den tidigare tränade TF-IDF-vektoriseraren
tfidf = joblib.load(
    "../data/processed/tfidf_vectorizer.joblib"
)

# Omvandla recensionerna till numeriska features
X_train = tfidf.transform(train_reviews["clean_text"])
X_test = tfidf.transform(test_reviews["clean_text"])

# Sentiment är det vi vill förutsäga
y_train = train_reviews["sentiment"]
y_test = test_reviews["sentiment"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


X_train: (80000, 10000)
X_test: (20000, 10000)


### Träna Multinomial Naive Bayes

In [ ]:
# Skapa Naive Bayes-modellen
nb_model = MultinomialNB()

# Träna modellen
nb_model.fit(X_train, y_train)

# Gör förutsägelser på testdata
nb_predictions = nb_model.predict(X_test)

print("Naive Bayes är tränad.")

Naive Bayes är tränad.


Modellen lär sig vilka ord som är vanliga i positiva, negativa och neutrala recensioner och använder det för att klassificera nya recensioner

### Resultat för Naive Bayes

In [7]:
# Beräkna resultat för Naive Bayes
nb_accuracy = accuracy_score(y_test, nb_predictions)
nb_f1 = f1_score(y_test, nb_predictions, average="macro")

print("Naive Bayes accuracy:", round(nb_accuracy, 3))
print("Naive Bayes Macro F1:", round(nb_f1, 3))


Naive Bayes accuracy: 0.844
Naive Bayes Macro F1: 0.571


Mäter hur bra Naive Bayes fungerar, så att man kan jämföra med den tidigare modellen

In [ ]:
# Samma modell som i 03.ipynb
logreg = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Träna på samma data som Naive Bayes
logreg.fit(X_train, y_train)

# Gör prediktioner på samma testdata
logreg_predictions = logreg.predict(X_test)

# Beräkna resultat
logreg_accuracy = accuracy_score(y_test, logreg_predictions)
logreg_f1 = f1_score(y_test, logreg_predictions, average="macro")

Tränar logistisk regression med samma träningsdata och testar båda modellerna på samma testdata.

### Modelljämförelse

In [9]:
# Sammanställ modellernas resultat
results = pd.DataFrame({
    "Modell": ["Logistisk regression", "Naive Bayes"],
    "Accuracy": [logreg_accuracy, nb_accuracy],
    "Macro F1": [logreg_f1, nb_f1]
})

# Visa jämförelsen
print(results.round(3).to_string(index=False))

              Modell  Accuracy  Macro F1
Logistisk regression     0.863     0.648
         Naive Bayes     0.844     0.571


Tabellen visar modellernas resultat så att vi enkelt kan jämföra dem.

### Slutsats

Vi har tränat en Multinomial Naive Bayes-modell och jämfört den med logistisk regression på samma träningsdata och testdata.

Logistisk regression fick 86,3 % accuracy och 0,648 i Macro F1, medan Naive Bayes fick 84,4 % accuracy och 0,571 i Macro F1.

Resultatet visar att logistisk regression presterade bättre än Naive Bayes på båda måtten. Därför är logistisk regression den bästa modellen av de två i vår jämförelse.